# Exercícios 1-5: Nível Iniciante — MongoDB + PyMongo

## Configuração da Conexão

In [ ]:
from pymongo import MongoClient
from statistics import mean
import re

# Ajuste a URI conforme seu ambiente
client = MongoClient("mongodb://localhost:27017/")
db = client["escola"]  # troque pelo nome do seu banco


## Exercício 1
**Listar todos os alunos ativos com idade maior que 20 anos, ordenados por nome.**

**mongosh:**
```js
db.alunos.find({ ativo: true, idade: { $gt: 20 } }).sort({ nome: 1 })
```


In [ ]:
# Exercício 1 - PyMongo
alunos = db.alunos.find(
    { "ativo": True, "idade": { "$gt": 20 } }
).sort("nome", 1)

for aluno in alunos:
    print(aluno)


## Exercício 2
**Contar quantos alunos existem por curso e mostrar apenas cursos com mais de 1 aluno.**

**mongosh:**
```js
db.alunos.aggregate([
  { $group: { _id: "$curso", total: { $sum: 1 } } },
  { $match: { total: { $gt: 1 } } }
])
```


In [ ]:
# Exercício 2 - PyMongo
def contar_alunos_por_curso():
    pipeline = [
        { "$group": { "_id": "$curso", "total": { "$sum": 1 } } },
        { "$match": { "total": { "$gt": 1 } } }
    ]
    resultado = db.alunos.aggregate(pipeline)
    return { doc["_id"]: doc["total"] for doc in resultado }

print(contar_alunos_por_curso())


## Exercício 3
**Encontrar todos os alunos cujo nome contém "Silva" (case-insensitive).**

**mongosh:**
```js
db.alunos.find({ nome: { $regex: "Silva", $options: "i" } })
```


In [ ]:
# Exercício 3 - PyMongo
padrao = re.compile("Silva", re.IGNORECASE)
alunos = db.alunos.find({ "nome": { "$regex": padrao } })

for aluno in alunos:
    print(aluno)


## Exercício 4
**Calcular a média geral de notas de todos os alunos ativos.**

**mongosh:**
```js
db.alunos.aggregate([
  { $match: { ativo: true } },
  { $unwind: "$notas" },
  { $group: { _id: null, media: { $avg: "$notas" } } }
])
```


In [ ]:
# Exercício 4 - PyMongo
alunos = db.alunos.find({ "ativo": True })
todas_notas = []

for aluno in alunos:
    todas_notas.extend(aluno.get("notas", []))

if todas_notas:
    media_geral = mean(todas_notas)
    print(f"Média geral: {media_geral:.2f}")
else:
    print("Nenhuma nota encontrada. Verifique se há alunos ativos com o campo 'notas' no banco.")

## Exercício 5
**Listar os 3 alunos com maior média individual, mostrando nome, curso e média.**

**mongosh:**
```js
db.alunos.aggregate([
  { $addFields: { media: { $avg: "$notas" } } },
  { $sort: { media: -1 } },
  { $limit: 3 },
  { $project: { nome: 1, curso: 1, media: 1 } }
])
```


In [ ]:
# Exercício 5 - PyMongo
alunos = list(db.alunos.find({}, { "nome": 1, "curso": 1, "notas": 1 }))

for aluno in alunos:
    notas = aluno.get("notas", [])
    aluno["media"] = sum(notas) / len(notas) if notas else 0

top3 = sorted(alunos, key=lambda x: x["media"], reverse=True)[:3]

for aluno in top3:
    print(f"Nome: {aluno['nome']} | Curso: {aluno['curso']} | Média: {aluno['media']:.2f}")
